<a href="https://colab.research.google.com/github/chavezaltamirano-ui/Growth-Models-in-Comparative/blob/main/1Growth_Models_in_Comparative_Perspective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Bloque 1 – PWT 11.0 (productividad y crecimiento)

In [17]:
# ============================================
# BLOQUE PWT 11.0 - Productividad y crecimiento
# China (CN) y Estados Unidos (US), 1990–2025
# ============================================

import pandas as pd
import os

os.makedirs('data_raw', exist_ok=True)
os.makedirs('data_clean', exist_ok=True)

# URL del archivo Stata de PWT 11.0 (Dataverse oficial)[web:334]
pwt_url_dta = 'https://dataverse.nl/api/access/datafile/554030'  # Stata file de PWT 11.0

# Leer archivo Stata
pwt_df = pd.read_stata(pwt_url_dta)

# Filtrar China y Estados Unidos, años 1990–2025
countries_names = ['China', 'United States']
pwt_china_us = pwt_df[pwt_df['country'].isin(countries_names)]
pwt_china_us = pwt_china_us[pwt_china_us['year'].between(1990, 2025)]

# Seleccionar variables de productividad y crecimiento[file:311]
cols_pwt = ['country', 'year', 'rgdpna', 'rkna', 'rtfpna', 'hc', 'pop']

missing_cols = [c for c in cols_pwt if c not in pwt_china_us.columns]
if missing_cols:
    print("ADVERTENCIA PWT: estas columnas no se encontraron:", missing_cols)
else:
    pwt_china_us = pwt_china_us[cols_pwt]

# Normalizar identificadores de país a códigos cortos (CN, US)
country_map_pwt = {
    'China': 'CN',
    'United States': 'US'
}
pwt_china_us['country'] = pwt_china_us['country'].map(country_map_pwt)

# Reordenar columnas
pwt_china_us = pwt_china_us[['country', 'year', 'rgdpna', 'rkna', 'rtfpna', 'hc', 'pop']]

print("\nVista previa PWT (China–US, 1990–2025):")
print(pwt_china_us.head())

# Guardar
pwt_raw_path = 'data_raw/pwt_china_us_1990_2025.csv'
pwt_clean_path = 'data_clean/pwt_china_us_1990_2025.csv'

pwt_china_us.to_csv(pwt_raw_path, index=False)
pwt_china_us.to_csv(pwt_clean_path, index=False)

print(f"\nPWT guardado en: {pwt_raw_path}")
print(f"PWT también copiado en: {pwt_clean_path}")


Vista previa PWT (China–US, 1990–2025):
     country  year       rgdpna      rkna    rtfpna        hc          pop
2482      CN  1990  1857144.750  0.036908  0.351905  1.956077  1153.582724
2483      CN  1991  2029162.375  0.039276  0.366076  1.991197  1170.788528
2484      CN  1992  2317807.250  0.043118  0.396038  2.026947  1184.574237
2485      CN  1993  2639603.750  0.048899  0.421519  2.063339  1197.308575
2486      CN  1994  2983718.250  0.055067  0.446154  2.100384  1209.003096

PWT guardado en: data_raw/pwt_china_us_1990_2025.csv
PWT también copiado en: data_clean/pwt_china_us_1990_2025.csv


Bloque 2 – WDI financiero (Banco Mundial)

In [18]:
# ============================================
# BLOQUE WDI FINANCIERO (World Bank)
# China (CN) y Estados Unidos (US), 1990–2025
# ============================================

import pandas as pd
import requests
import os

os.makedirs('data_raw', exist_ok=True)
os.makedirs('data_clean', exist_ok=True)

def download_wdi_indicator(indicator_code, countries, start_year=1990, end_year=2025):
    """
    Descarga un indicador WDI para una lista de países y un rango de años.
    Usa la API del Banco Mundial (JSON).[web:316]
    Devuelve un DataFrame con columnas: country, year, indicator_code.
    """
    frames = []
    for country in countries:
        url = (
            f"http://api.worldbank.org/v2/country/{country}/indicator/{indicator_code}"
            f"?date={start_year}:{end_year}&format=json&per_page=20000"
        )
        resp = requests.get(url)
        if resp.status_code != 200:
            print(f"Error HTTP para {indicator_code}, país {country}: {resp.status_code}")
            continue

        data = resp.json()
        if len(data) < 2 or data[1] is None:
            print(f"Sin datos para {indicator_code}, país {country}")
            continue

        rows = []
        for entry in data[1]:
            year = entry.get('date')
            value = entry.get('value')
            try:
                year_int = int(year)
            except (TypeError, ValueError):
                continue
            rows.append({'country': country, 'year': year_int, indicator_code: value})

        if rows:
            frames.append(pd.DataFrame(rows))

    if frames:
        df = pd.concat(frames, ignore_index=True)
        return df
    else:
        return pd.DataFrame(columns=['country', 'year', indicator_code])

countries = ['CN', 'US']
start_year, end_year = 1990, 2025

indicators_financial = [
    'FS.AST.PRVT.GD.ZS',     # Crédito al sector privado (% PIB)
    'FS.AST.DOMS.GD.ZS',     # Crédito del sector financiero (% PIB)
    'BX.TRF.PWKR.DT.GD.ZS'   # Remesas (% PIB)[web:316]
]

df_fin = None
for ind in indicators_financial:
    print(f"Descargando indicador financiero WDI: {ind}")
    df_ind = download_wdi_indicator(ind, countries, start_year, end_year)
    if df_fin is None:
        df_fin = df_ind
    else:
        df_fin = df_fin.merge(df_ind, on=['country', 'year'], how='outer')

rename_fin = {
    'FS.AST.PRVT.GD.ZS': 'credit_priv_gdp',
    'FS.AST.DOMS.GD.ZS': 'credit_finsec_gdp',
    'BX.TRF.PWKR.DT.GD.ZS': 'remittances_gdp'
}
df_fin = df_fin.rename(columns=rename_fin)
df_fin = df_fin.sort_values(['country', 'year'])

print("\nVista previa WDI financiero:")
print(df_fin.head())

wdi_fin_raw = 'data_raw/wdi_fin_china_us_1990_2025.csv'
wdi_fin_clean = 'data_clean/wdi_fin_china_us_1990_2025.csv'
df_fin.to_csv(wdi_fin_raw, index=False)
df_fin.to_csv(wdi_fin_clean, index=False)

print(f"\nWDI FIN guardado en: {wdi_fin_raw}")
print(f"WDI FIN también copiado en: {wdi_fin_clean}")

Descargando indicador financiero WDI: FS.AST.PRVT.GD.ZS
Descargando indicador financiero WDI: FS.AST.DOMS.GD.ZS
Descargando indicador financiero WDI: BX.TRF.PWKR.DT.GD.ZS


/tmp/ipykernel_424/2570008970.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(frames, ignore_index=True)



Vista previa WDI financiero:
  country  year  credit_priv_gdp  credit_finsec_gdp  remittances_gdp
0      CN  1990        86.033020                NaN         0.054193
1      CN  1991        88.101074                NaN         0.101381
2      CN  1992        86.053063                NaN         0.144545
3      CN  1993        96.506392                NaN         0.141388
4      CN  1994        85.487555                NaN         0.151165

WDI FIN guardado en: data_raw/wdi_fin_china_us_1990_2025.csv
WDI FIN también copiado en: data_clean/wdi_fin_china_us_1990_2025.csv


Bloque 3 – WDI macro (Banco Mundial)

In [19]:
# ============================================
# BLOQUE WDI MACRO (World Bank)
# China (CN) y Estados Unidos (US), 1990–2025
# PIB, crecimiento, inversión, inflación, apertura, gasto, desempleo, urbanización
# ============================================

import pandas as pd
import requests
import os

os.makedirs('data_raw', exist_ok=True)
os.makedirs('data_clean', exist_ok=True)

# Reutilizamos download_wdi_indicator de la celda anterior
def download_wdi_indicator(indicator_code, countries, start_year=1990, end_year=2025):
    frames = []
    for country in countries:
        url = (
            f"http://api.worldbank.org/v2/country/{country}/indicator/{indicator_code}"
            f"?date={start_year}:{end_year}&format=json&per_page=20000"
        )
        resp = requests.get(url)
        if resp.status_code != 200:
            print(f"Error HTTP para {indicator_code}, país {country}: {resp.status_code}")
            continue

        data = resp.json()
        if len(data) < 2 or data[1] is None:
            print(f"Sin datos para {indicator_code}, país {country}")
            continue

        rows = []
        for entry in data[1]:
            year = entry.get('date')
            value = entry.get('value')
            try:
                year_int = int(year)
            except (TypeError, ValueError):
                continue
            rows.append({'country': country, 'year': year_int, indicator_code: value})

        if rows:
            frames.append(pd.DataFrame(rows))

    if frames:
        df = pd.concat(frames, ignore_index=True)
        return df
    else:
        return pd.DataFrame(columns=['country', 'year', indicator_code])

countries = ['CN', 'US']
start_year, end_year = 1990, 2025

indicators_macro = [
    "NY.GDP.MKTP.KD",      # PIB real (constant 2015 US$)[web:312][web:319]
    "NY.GDP.MKTP.KD.ZG",   # Crecimiento del PIB (%)[web:314]
    "NE.GDI.FTOT.ZS",      # GFCF % PIB[file:311]
    "NE.GDI.FTOT.KD",      # GFCF constante US$
    "FP.CPI.TOTL.ZG",      # Inflación CPI (%)[web:315]
    "NE.TRD.GNFS.ZS",      # Comercio % PIB
    "NE.CON.GOVT.ZS",      # Consumo gobierno % PIB
    "SL.UEM.TOTL.ZS",      # Desempleo %
    "SP.URB.TOTL.IN.ZS",   # Urbanización %
]

df_wdi_macro = None
for ind in indicators_macro:
    print(f"Descargando indicador macro WDI: {ind}")
    df_ind = download_wdi_indicator(ind, countries, start_year, end_year)
    if df_wdi_macro is None:
        df_wdi_macro = df_ind
    else:
        df_wdi_macro = df_wdi_macro.merge(df_ind, on=['country', 'year'], how='outer')

rename_macro = {
    "NY.GDP.MKTP.KD": "gdp_const_usd",
    "NY.GDP.MKTP.KD.ZG": "gdp_growth_pct",
    "NE.GDI.FTOT.ZS": "inv_gfcf_gdp",
    "NE.GDI.FTOT.KD": "inv_gfcf_const_usd",
    "FP.CPI.TOTL.ZG": "inflation_cpi_pct",
    "NE.TRD.GNFS.ZS": "trade_gdp",
    "NE.CON.GOVT.ZS": "gov_cons_gdp",
    "SL.UEM.TOTL.ZS": "unemployment_pct",
    "SP.URB.TOTL.IN.ZS": "urban_pop_pct",
}
df_wdi_macro = df_wdi_macro.rename(columns=rename_macro)
df_wdi_macro = df_wdi_macro.sort_values(["country", "year"])

print("\nVista previa WDI macro:")
print(df_wdi_macro.head())

wdi_macro_raw = "data_raw/wdi_macro_china_us_1990_2025.csv"
wdi_macro_clean = "data_clean/wdi_macro_china_us_1990_2025.csv"
df_wdi_macro.to_csv(wdi_macro_raw, index=False)
df_wdi_macro.to_csv(wdi_macro_clean, index=False)

print(f"\nWDI MACRO guardado en: {wdi_macro_raw}")
print(f"WDI MACRO también copiado en: {wdi_macro_clean}")

Descargando indicador macro WDI: NY.GDP.MKTP.KD
Descargando indicador macro WDI: NY.GDP.MKTP.KD.ZG
Descargando indicador macro WDI: NE.GDI.FTOT.ZS
Descargando indicador macro WDI: NE.GDI.FTOT.KD
Descargando indicador macro WDI: FP.CPI.TOTL.ZG
Descargando indicador macro WDI: NE.TRD.GNFS.ZS
Descargando indicador macro WDI: NE.CON.GOVT.ZS
Descargando indicador macro WDI: SL.UEM.TOTL.ZS
Descargando indicador macro WDI: SP.URB.TOTL.IN.ZS

Vista previa WDI macro:
  country  year  gdp_const_usd  gdp_growth_pct  inv_gfcf_gdp  \
0      CN  1990   1.041178e+12        3.922520     23.942156   
1      CN  1991   1.138795e+12        9.375632     26.033374   
2      CN  1992   1.301616e+12       14.297623     30.615923   
3      CN  1993   1.482880e+12       13.926103     37.310754   
4      CN  1994   1.676881e+12       13.082712     34.694508   

   inv_gfcf_const_usd  inflation_cpi_pct  trade_gdp  gov_cons_gdp  \
0                 NaN           3.052290  24.225982     13.824195   
1             

Bloque 4 – BIS (extracto trimestral + anual)

In [22]:
# ============================================
# BLOQUE BIS - Crédito al sector no financiero
# Extracto trimestral + anual CN/US 1990–2025
# ============================================

import pandas as pd
import requests
import os

os.makedirs('data_raw', exist_ok=True)
os.makedirs('data_clean', exist_ok=True)

bis_xlsx_path = "data_raw/totcredit.xlsx"

# Descargar el Excel BIS si no existe
if not os.path.exists(bis_xlsx_path):
    bis_xlsx_url = "https://www.bis.org/statistics/totcredit/totcredit.xlsx"
    print("Descargando totcredit.xlsx desde BIS...")
    resp_bis = requests.get(bis_xlsx_url)
    if resp_bis.status_code != 200:
        raise RuntimeError(f"Error al descargar totcredit.xlsx: HTTP {resp_bis.status_code}")
    with open(bis_xlsx_path, "wb") as f:
        f.write(resp_bis.content)
    print("Descarga BIS completada.")
else:
    print("totcredit.xlsx ya existe en data_raw; usando el existente.")

# Leer 'Quarterly Series'
xls = pd.ExcelFile(bis_xlsx_path)
print("Hojas disponibles en totcredit.xlsx:", xls.sheet_names)

sheet_q = "Quarterly Series"
df_q_raw = pd.read_excel(bis_xlsx_path, sheet_name=sheet_q, header=None)

print("\nDimensiones 'Quarterly Series':", df_q_raw.shape)
print("Primeras filas (referencia):")
print(df_q_raw.iloc[:5, :10])

# Metadatos de columnas
meta_list = []
for j in range(1, df_q_raw.shape[1]):
    header = df_q_raw.iloc[0, j]
    unit   = df_q_raw.iloc[1, j]
    area   = df_q_raw.iloc[2, j]
    code   = df_q_raw.iloc[3, j]
    meta_list.append({
        "col_index": j,
        "header": str(header),
        "unit": str(unit),
        "area": str(area),
        "code": str(code),
    })
meta = pd.DataFrame(meta_list)

mask_china = meta["header"].str.contains("China", case=False, na=False)
mask_us    = meta["header"].str.contains("United States", case=False, na=False)
mask_pct   = meta["header"].str.contains("Percentage of GDP", case=False, na=False)

meta_china_pct = meta[mask_china & mask_pct]
meta_us_pct    = meta[mask_us & mask_pct]

print("\nColumnas BIS China (% PIB):")
print(meta_china_pct)

print("\nColumnas BIS US (% PIB):")
print(meta_us_pct)

# Fechas y extracción de series
date_raw = df_q_raw.iloc[:, 0]
mask_date = ~date_raw.isna() & \
            ~date_raw.astype(str).str.contains("Back to menu", case=False, na=False) & \
            ~date_raw.astype(str).str.contains("Period", case=False, na=False)

date_clean = date_raw[mask_date]
dates = pd.to_datetime(date_clean, errors="coerce")
valid_idx = date_clean.index

df_bis_extracted = pd.DataFrame({"date": dates})

# Añadir columnas China
for _, row in meta_china_pct.iterrows():
    j = row["col_index"]
    col_name = f"CN_{row['code']}"
    values = df_q_raw.iloc[valid_idx, j].reset_index(drop=True)
    df_bis_extracted[col_name] = values

# Añadir columnas Estados Unidos
for _, row in meta_us_pct.iterrows():
    j = row["col_index"]
    col_name = f"US_{row['code']}"
    values = df_q_raw.iloc[valid_idx, j].reset_index(drop=True)
    df_bis_extracted[col_name] = values

print("\nVista previa BIS extracto (trimestral):")
print(df_bis_extracted.head())

bis_extract_raw = "data_raw/bis_totcredit_extract_cn_us.csv"
bis_extract_clean = "data_clean/bis_totcredit_extract_cn_us.csv"
df_bis_extracted.to_csv(bis_extract_raw, index=False)
df_bis_extracted.to_csv(bis_extract_clean, index=False)

print(f"\nBIS EXTRACTO guardado en: {bis_extract_raw}")
print(f"BIS EXTRACTO también copiado en: {bis_extract_clean}")

# ---- BIS anual ----

extract_path = bis_extract_clean if os.path.exists(bis_extract_clean) else bis_extract_raw
df_q = pd.read_csv(extract_path)

df_q["date"] = pd.to_datetime(df_q["date"], errors="coerce")
df_q = df_q.dropna(subset=["date"])
df_q["year"] = df_q["date"].dt.year
df_q = df_q[(df_q["year"] >= 1990) & (df_q["year"] <= 2025)]

col_map = {
    # China
    "CN_Q:CN:C:A:M:770:A": ("CN", "bis_tot_credit_gdp"),
    "CN_Q:CN:P:A:M:770:A": ("CN", "bis_pvt_credit_gdp"),
    "CN_Q:CN:H:A:M:770:A": ("CN", "bis_hh_credit_gdp"),
    "CN_Q:CN:N:A:M:770:A": ("CN", "bis_nfc_credit_gdp"),
    "CN_Q:CN:G:A:N:770:A": ("CN", "bis_gov_credit_gdp"),
    "CN_Q:CN:P:B:M:770:A": ("CN", "bis_pvt_banks_credit_gdp"),

    # Estados Unidos
    "US_Q:US:C:A:M:770:A": ("US", "bis_tot_credit_gdp"),
    "US_Q:US:P:A:M:770:A": ("US", "bis_pvt_credit_gdp"),
    "US_Q:US:H:A:M:770:A": ("US", "bis_hh_credit_gdp"),
    "US_Q:US:N:A:M:770:A": ("US", "bis_nfc_credit_gdp"),
    "US_Q:US:G:A:N:770:A": ("US", "bis_gov_credit_gdp"),
    "US_Q:US:P:B:M:770:A": ("US", "bis_pvt_banks_credit_gdp"),
}

records = []
for col in df_q.columns:
    if col in ["date", "year"]:
        continue
    if col not in col_map:
        continue

    country, series_short = col_map[col]
    series_values = df_q[col].values
    dates = df_q["date"].values
    years = df_q["year"].values

    for dt, yr, val in zip(dates, years, series_values):
        records.append({
            "country": country,
            "year": yr,
            "date": dt,
            "series_short": series_short,
            "value": val,
        })

df_long = pd.DataFrame(records)
df_long = (
    df_long
    .sort_values(["country", "series_short", "year", "date"])
    .groupby(["country", "series_short", "year"], as_index=False)
    .tail(1)
)

df_bis_annual = (
    df_long
    .pivot(index=["country", "year"], columns="series_short", values="value")
    .reset_index()
)
df_bis_annual = df_bis_annual.sort_values(["country", "year"])

print("\nVista previa BIS anual:")
print(df_bis_annual.head())

bis_annual_raw = "data_raw/bis_totcredit_cn_us_1990_2025.csv"
bis_annual_clean = "data_clean/bis_totcredit_cn_us_1990_2025.csv"
df_bis_annual.to_csv(bis_annual_raw, index=False)
df_bis_annual.to_csv(bis_annual_clean, index=False)

print(f"\nBIS ANUAL guardado en: {bis_annual_raw}")
print(f"BIS ANUAL también copiado en: {bis_annual_clean}")

totcredit.xlsx ya existe en data_raw; usando el existente.
Hojas disponibles en totcredit.xlsx: ['Content', 'Summary Documentation', 'Quarterly Series']

Dimensiones 'Quarterly Series': (337, 1134)
Primeras filas (referencia):
                     0                                                  1  \
0         Back to menu  Emerging market economies (aggregate) - Credit...   
1                  NaN                                   Per cent (Units)   
2                  NaN              Emerging market economies (aggregate)   
3               Period                                   Q:4T:C:A:M:770:A   
4  1940-06-30 00:00:00                                                NaN   

                                                   2  \
0  Emerging market economies (aggregate) - Credit...   
1                                   Per cent (Units)   
2              Emerging market economies (aggregate)   
3                                   Q:4T:C:A:M:799:A   
4                             

Bloque 5 – Merge maestro

In [24]:
# ============================================
# BLOQUE MERGE MAESTRO
# Panel CN/US 1990–2025: PWT + WDI macro + WDI financiero + BIS
# ============================================

import pandas as pd
import os

pwt_path       = "data_clean/pwt_china_us_1990_2025.csv"
wdi_macro_path = "data_clean/wdi_macro_china_us_1990_2025.csv"
wdi_fin_path   = "data_clean/wdi_fin_china_us_1990_2025.csv"
bis_path       = "data_clean/bis_totcredit_cn_us_1990_2025.csv"

for path in [pwt_path, wdi_macro_path, wdi_fin_path, bis_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"No se encontró el archivo esperado: {path}")

df_pwt       = pd.read_csv(pwt_path)
df_wdi_macro = pd.read_csv(wdi_macro_path)
df_wdi_fin   = pd.read_csv(wdi_fin_path)
df_bis       = pd.read_csv(bis_path)

for df in [df_pwt, df_wdi_macro, df_wdi_fin, df_bis]:
    df.sort_values(["country", "year"], inplace=True)

df_panel = df_pwt.copy()
df_panel = df_panel.merge(df_wdi_macro, on=["country", "year"], how="left")
df_panel = df_panel.merge(df_wdi_fin,   on=["country", "year"], how="left")
df_panel = df_panel.merge(df_bis,       on=["country", "year"], how="left")

df_panel = df_panel.sort_values(["country", "year"])

print("\nVista rápida del panel combinado (primeras filas):")
print(df_panel.head())

print("\nVista rápida del panel combinado (últimas filas):")
print(df_panel.tail())

print("\nNúmero de columnas en el panel:", df_panel.shape[1])

panel_raw_path   = "data_raw/china_us_panel_1990_2025.csv"
panel_clean_path = "data_clean/china_us_panel_1990_2025.csv"

df_panel.to_csv(panel_raw_path, index=False)
df_panel.to_csv(panel_clean_path, index=False)

print(f"\nPANEL guardado en: {panel_raw_path}")
print(f"PANEL también copiado en: {panel_clean_path}")


Vista rápida del panel combinado (primeras filas):
  country  year     rgdpna      rkna    rtfpna        hc          pop  \
0      CN  1990  1857144.8  0.036908  0.351905  1.956077  1153.582724   
1      CN  1991  2029162.4  0.039276  0.366076  1.991197  1170.788528   
2      CN  1992  2317807.2  0.043118  0.396038  2.026947  1184.574237   
3      CN  1993  2639603.8  0.048899  0.421519  2.063338  1197.308575   
4      CN  1994  2983718.2  0.055067  0.446154  2.100384  1209.003096   

   gdp_const_usd  gdp_growth_pct  inv_gfcf_gdp  ...  urban_pop_pct  \
0   1.041178e+12        3.922520     23.942156  ...      26.195702   
1   1.138795e+12        9.375632     26.033374  ...      26.940247   
2   1.301616e+12       14.297623     30.615923  ...      27.459869   
3   1.482880e+12       13.926103     37.310754  ...      27.990076   
4   1.676881e+12       13.082712     34.694508  ...      28.509804   

   credit_priv_gdp  credit_finsec_gdp  remittances_gdp  bis_gov_credit_gdp  \
0        8